In [1]:
# Cell 0 — install (run once)
!pip install pyarrow pandas


Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Cell 1 — Task 1: Create an Arrow Table
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyarrow.ipc as ipc
import pandas as pd

data = {
    "employee_id": [1, 2, 3, 4, 5, 6],
    "name": ["Asha", "Rahul", "Neha", "Vikram", "Priya", "Arjun"],
    "department": ["IT", "HR", "IT", "Finance", "HR", "Finance"],
    "salary": [60000, 45000, 70000, 55000, 48000, 65000],
    "city": ["Delhi", "Mumbai", "Bengaluru", "Delhi", "Mumbai", "Chennai"],
}

employee_table = pa.table(data)
print(employee_table)


pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
----
employee_id: [[1,2,3,4,5,6]]
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]
city: [["Delhi","Mumbai","Bengaluru","Delhi","Mumbai","Chennai"]]


In [3]:
# Cell 2 — Task 2: Schema
print(employee_table.schema)
# Answers: employee_id -> int64, name -> string, salary -> int64


employee_id: int64
name: string
department: string
salary: int64
city: string


In [4]:
# Cell 3 — Task 3: Inspect
print("Rows:", employee_table.num_rows)
print("Columns:", employee_table.num_columns)
print("Column names:", employee_table.column_names)
print(employee_table.column("name"))
print(employee_table.slice(0, 3))


Rows: 6
Columns: 5
Column names: ['employee_id', 'name', 'department', 'salary', 'city']
[
  [
    "Asha",
    "Rahul",
    "Neha",
    "Vikram",
    "Priya",
    "Arjun"
  ]
]
pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
----
employee_id: [[1,2,3]]
name: [["Asha","Rahul","Neha"]]
department: [["IT","HR","IT"]]
salary: [[60000,45000,70000]]
city: [["Delhi","Mumbai","Bengaluru"]]


In [5]:
# Cell 4 — Task 4: Select columns
selected_table = employee_table.select(["name", "department", "salary"])
print(selected_table)


pyarrow.Table
name: string
department: string
salary: int64
----
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]


In [6]:
# Cell 4 — Task 4: Select columns
selected_table = employee_table.select(["name", "department", "salary"])
print(selected_table)


pyarrow.Table
name: string
department: string
salary: int64
----
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]


In [7]:
# Cell 5 — Task 5: Filter salary > 50000
high_salary_table = employee_table.filter(pc.greater(employee_table["salary"], 50000))
print(high_salary_table)


pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
----
employee_id: [[1,3,4,6]]
name: [["Asha","Neha","Vikram","Arjun"]]
department: [["IT","IT","Finance","Finance"]]
salary: [[60000,70000,55000,65000]]
city: [["Delhi","Bengaluru","Delhi","Chennai"]]


In [8]:
# Cell 6 — Task 6: IT department
it_employees = employee_table.filter(pc.equal(employee_table["department"], "IT"))
print(it_employees)


pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
----
employee_id: [[1,3]]
name: [["Asha","Neha"]]
department: [["IT","IT"]]
salary: [[60000,70000]]
city: [["Delhi","Bengaluru"]]


In [9]:
# Cell 7 — Task 7: Calculations
salary_column = employee_table["salary"]
print("Average salary:", pc.mean(salary_column).as_py())
print("Maximum salary:", pc.max(salary_column).as_py())
print("Minimum salary:", pc.min(salary_column).as_py())
print("Total salary:", pc.sum(salary_column).as_py())


Average salary: 57166.666666666664
Maximum salary: 70000
Minimum salary: 45000
Total salary: 343000


In [10]:
# Cell 8 — Task 8: Add bonus column (10% of salary)
employee_table = employee_table.append_column(
    "bonus", pc.multiply(employee_table["salary"], 0.10)
)
print(employee_table)


pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
bonus: double
----
employee_id: [[1,2,3,4,5,6]]
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]
city: [["Delhi","Mumbai","Bengaluru","Delhi","Mumbai","Chennai"]]
bonus: [[6000,4500,7000,5500,4800,6500]]


In [11]:
# Cell 9 — Task 9 & 10: Arrow <-> Pandas
employee_df = employee_table.to_pandas()
print(employee_df)

new_arrow_table = pa.Table.from_pandas(employee_df, preserve_index=False)
print(new_arrow_table)


   employee_id    name department  salary       city   bonus
0            1    Asha         IT   60000      Delhi  6000.0
1            2   Rahul         HR   45000     Mumbai  4500.0
2            3    Neha         IT   70000  Bengaluru  7000.0
3            4  Vikram    Finance   55000      Delhi  5500.0
4            5   Priya         HR   48000     Mumbai  4800.0
5            6   Arjun    Finance   65000    Chennai  6500.0
pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
bonus: double
----
employee_id: [[1,2,3,4,5,6]]
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]
city: [["Delhi","Mumbai","Bengaluru","Delhi","Mumbai","Chennai"]]
bonus: [[6000,4500,7000,5500,4800,6500]]


In [12]:
# Cell 10 — Task 11 & 12: Parquet write/read
pq.write_table(employee_table, "employees.parquet")
print("Parquet file created successfully.")

loaded_table = pq.read_table("employees.parquet")
print(loaded_table)


Parquet file created successfully.
pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
bonus: double
----
employee_id: [[1,2,3,4,5,6]]
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]
city: [["Delhi","Mumbai","Bengaluru","Delhi","Mumbai","Chennai"]]
bonus: [[6000,4500,7000,5500,4800,6500]]


In [13]:
# Cell 11 — Task 13 & 14: Arrow IPC write/read
with ipc.new_file("employees.arrow", employee_table.schema) as writer:
    writer.write_table(employee_table)
print("Arrow IPC file created successfully.")

with ipc.open_file("employees.arrow") as reader:
    ipc_table = reader.read_all()
print(ipc_table)


Arrow IPC file created successfully.
pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
bonus: double
----
employee_id: [[1,2,3,4,5,6]]
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]
city: [["Delhi","Mumbai","Bengaluru","Delhi","Mumbai","Chennai"]]
bonus: [[6000,4500,7000,5500,4800,6500]]


In [14]:
# Cell 12 — Bonus tasks
# 1. Employees in Delhi
delhi = employee_table.filter(pc.equal(employee_table["city"], "Delhi"))
print("Delhi employees:\n", delhi.to_pandas())

# 2. Salary between 50000 and 65000
between = employee_table.filter(
    pc.and_(
        pc.greater_equal(employee_table["salary"], 50000),
        pc.less_equal(employee_table["salary"], 65000),
    )
)
print("\nSalary 50k-65k:\n", between.to_pandas())

# 3. annual_salary column
employee_table = employee_table.append_column(
    "annual_salary", pc.multiply(employee_table["salary"], 12)
)
print("\nWith annual_salary:\n", employee_table.to_pandas())

# 4. Save only IT employees
it_only = employee_table.filter(pc.equal(employee_table["department"], "IT"))
pq.write_table(it_only, "it_employees.parquet")
print("\nit_employees.parquet created.")

# 5. Read only name and salary from Parquet
selected_columns = pq.read_table("employees.parquet", columns=["name", "salary"])
print("\nSelected columns:\n", selected_columns.to_pandas())

# 6. Sort by salary (highest first)
sorted_table = employee_table.sort_by([("salary", "descending")])
print("\nSorted by salary:\n", sorted_table.to_pandas())


Delhi employees:
    employee_id    name department  salary   city   bonus
0            1    Asha         IT   60000  Delhi  6000.0
1            4  Vikram    Finance   55000  Delhi  5500.0

Salary 50k-65k:
    employee_id    name department  salary     city   bonus
0            1    Asha         IT   60000    Delhi  6000.0
1            4  Vikram    Finance   55000    Delhi  5500.0
2            6   Arjun    Finance   65000  Chennai  6500.0

With annual_salary:
    employee_id    name department  salary       city   bonus  annual_salary
0            1    Asha         IT   60000      Delhi  6000.0         720000
1            2   Rahul         HR   45000     Mumbai  4500.0         540000
2            3    Neha         IT   70000  Bengaluru  7000.0         840000
3            4  Vikram    Finance   55000      Delhi  5500.0         660000
4            5   Priya         HR   48000     Mumbai  4800.0         576000
5            6   Arjun    Finance   65000    Chennai  6500.0         780000

it